# Day 2 Lab — Anthropic (Claude) edition

Day 2 is about understanding what an LLM API call *actually is* underneath the client library. The lesson is the same with Claude; only the endpoint and the call shape change.

We'll: (1) hit the raw HTTP endpoint by hand, (2) see that the `anthropic` package is just a thin wrapper around that, then (3) cover the "OpenAI-compatible endpoints" idea — which still applies to Gemini and Ollama exactly as in the original, since those parts don't use your Anthropic key.

## First — the Messages API

1. The simplest way to call Claude.
2. You hand the model a conversation and it predicts the next assistant turn.
3. OpenAI popularized the "Chat Completions" shape; Anthropic's equivalent is the **Messages API**, which is very similar but has a few deliberate differences (system prompt is separate, `max_tokens` is required).

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('ANTHROPIC_API_KEY')

if not api_key:
    print("No API key was found - add ANTHROPIC_API_KEY=sk-ant-... to your .env file")
elif not api_key.startswith("sk-ant-"):
    print("An API key was found, but it doesn't start sk-ant-; please check you're using your Anthropic key")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


## Do you know what an Endpoint is?

An endpoint is just a URL you send an HTTP request to. Here's Anthropic's. Note the differences from OpenAI's raw call:

- URL is `https://api.anthropic.com/v1/messages`
- Auth header is `x-api-key` (not `Authorization: Bearer`)
- You must send an `anthropic-version` header
- `max_tokens` is required in the body

In [2]:
import requests

headers = {
    "x-api-key": api_key,
    "anthropic-version": "2023-06-01",
    "Content-Type": "application/json",
}

payload = {
    "model": "claude-haiku-4-5",
    "max_tokens": 1000,
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}
    ],
}

payload

{'model': 'claude-haiku-4-5',
 'max_tokens': 1000,
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

In [3]:
response = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers=headers,
    json=payload,
)

response.json()

{'model': 'claude-haiku-4-5-20251001',
 'id': 'msg_01Xx3R6W8ZxBeAPy9RLfKyAc',
 'type': 'message',
 'role': 'assistant',
 'content': [{'type': 'text',
   'text': "# Here's a fun fact:\n\nHoney never spoils! Archaeologists have found 3,000-year-old honey in Egyptian tombs that was still perfectly edible. This is because honey's low moisture content and acidic pH make it an inhospitable environment for bacteria and mold. 🍯"}],
 'stop_reason': 'end_turn',
 'stop_sequence': None,
 'stop_details': None,
 'usage': {'input_tokens': 12,
  'cache_creation_input_tokens': 0,
  'cache_read_input_tokens': 0,
  'cache_creation': {'ephemeral_5m_input_tokens': 0,
   'ephemeral_1h_input_tokens': 0},
  'output_tokens': 71,
  'service_tier': 'standard',
  'inference_geo': 'not_available'}}

In [4]:
# Note the response shape differs from OpenAI: the text lives in content[0]["text"]
response.json()["content"][0]["text"]

"# Here's a fun fact:\n\nHoney never spoils! Archaeologists have found 3,000-year-old honey in Egyptian tombs that was still perfectly edible. This is because honey's low moisture content and acidic pH make it an inhospitable environment for bacteria and mold. 🍯"

# What is the anthropic package?

It's a **Python client library** — nothing more than a wrapper around making the exact HTTP call you just made by hand. It lets you write clean Python instead of juggling raw JSON and headers. It's open-source and lightweight; it does **not** contain any model — the model lives on Anthropic's servers.

In [ ]:
# Create the Anthropic client - it reads ANTHROPIC_API_KEY from your environment automatically

import anthropic
client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1000,
    messages=[{"role": "user", "content": "Tell me a fun fact"}],
)

response.content[0].text

## The "OpenAI-compatible endpoint" idea

OpenAI's Chat Completions shape became so common that other providers exposed identical endpoints, so you can reuse the `openai` client just by pointing it at a different `base_url` and key. The two cells below show this with **Gemini** and **Ollama** — neither uses your Anthropic key, so they're unchanged from the original course.

(Worth knowing: Anthropic *also* offers an OpenAI-compatible endpoint at `https://api.anthropic.com/v1/` if you ever want to call Claude through the `openai` client. But the native `anthropic` package above is the recommended way.)

### OPTIONAL — Gemini

If you want to try Gemini, get a key at https://aistudio.google.com/api-keys and add `GOOGLE_API_KEY=AIz...` to your `.env`. Otherwise skip the next two cells.

In [ ]:
from openai import OpenAI

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No Google key found - skip the next cell, or add GOOGLE_API_KEY to .env")
elif not google_api_key.startswith("AIz"):
    print("A key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")

In [ ]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[{"role": "user", "content": "Tell me a fun fact"}],
)
response.choices[0].message.content

## Ollama also gives an OpenAI-compatible endpoint — running locally on your machine, free

If the next cell doesn't print that Ollama is running, open a terminal and run `ollama serve`.

In [ ]:
requests.get("http://localhost:11434").content

### Download llama3.2 from Meta

Use `llama3.2:1b` if your machine is smaller. Don't grab llama3.3 or llama4 — too big for a laptop.

In [ ]:
!ollama pull llama3.2

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [ ]:
response = ollama.chat.completions.create(
    model="llama3.2",
    messages=[{"role": "user", "content": "Tell me a fun fact"}],
)
response.choices[0].message.content

In [ ]:
# deepseek-r1:1.5b - DeepSeek distilled into Qwen from Alibaba Cloud
!ollama pull deepseek-r1:1.5b

In [ ]:
response = ollama.chat.completions.create(
    model="deepseek-r1:1.5b",
    messages=[{"role": "user", "content": "Tell me a fun fact"}],
)
response.choices[0].message.content

# HOMEWORK EXERCISE

Upgrade your Day 1 summarizer to use a local open-source model via Ollama instead of a paid API.

**Benefits:** no API charges; data never leaves your machine.
**Trade-off:** noticeably less capable than a frontier model like Claude.

Since you're on the Anthropic track, you now have two free/cheap ways to do Day 1: Claude Haiku (cheap, very capable) or Ollama (free, local). Try the summarizer with `ollama.chat.completions.create(...)` using `llama3.2`, swapping it in for the Anthropic call.